In [16]:
import pandas as pd
import pickle
import sys
#print(sys.version)
#print(sys.executable)

In [8]:
# Load the pickle files
pipedir = '/home/bernthold.17/SURP2025/'
rmz_gmi_spline_file = open(f"{pipedir}/Transforms/psrmz_psgmi_transform.pickle", 'rb')
rmz_imz_spline_file = open(f"{pipedir}/Transforms/psrmz_psimz_transform.pickle", 'rb')

rmz_gmi_spline = pickle.load(rmz_gmi_spline_file)
rmz_imz_spline = pickle.load(rmz_imz_spline_file)

rmz_gmi_spline_file.close()
rmz_imz_spline_file.close()

/tmp/slurmtmp.8734792/ipykernel_3337717/2278967040.py:6: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  rmz_gmi_spline = pickle.load(rmz_gmi_spline_file)
/tmp/slurmtmp.8734792/ipykernel_3337717/2278967040.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to acce

In [10]:
# Read in the MISHAPS file using Pandas
contents = pd.read_csv('/home/bernthold.17/SURP2025/Data_Files/MISHAPS_F1_DITH0_N10.cmd.calibrate_ps', header = None, sep = '\\s+')

# Create header titles for every column in the file
cmd_labels = ['ID', 'RA_z', 'Dec_z', 'ID_z', 'X_z', 'Y_z', 'mis_z', 'mis_z_err', 'Sky_z', 'qflag_z', '?1z', '?2z',
       'RA_r', 'Dec_r', 'ID_r', 'X_r', 'Y_r', 'mis_r', 'mis_r_err', 'Sky_r', 'qflag_r', '?1r', '?2r',
       'RA_u', 'Dec_u', 'ID_u', 'X_u', 'Y_u', 'mis_u', 'mis_u_err', 'Sky_u', 'qflag_u', '?1u', '?2u',
       'RA_g', 'Dec_g', 'ID_g', 'X_g', 'Y_g', 'mis_g', 'mis_g_err', 'Sky_g', 'qflag_g', '?1g', '?2g',
       'RA_i', 'Dec_i', 'ID_i', 'X_i', 'Y_i', 'mis_i', 'mis_i_err', 'Sky_i', 'qflag_i', '?1i', '?2i',
       'RA_Y', 'Dec_Y', 'ID_Y', 'X_Y', 'Y_Y', 'mis_Y', 'mis_Y_err', 'Sky_Y', 'qflag_Y', '?1Y', '?2Y']
              
# Apply those header titles to the file
contents.columns = cmd_labels

In [11]:
# Filter
mask_flags = (contents['qflag_z'] == 1) & (contents['qflag_r'] == 1)
mask_err = (contents['mis_r_err'] < 0.2) & (contents['mis_z_err'] < 0.2)
mask_sky = (contents["Sky_r"] < 800) & (contents["Sky_z"] < 3500)

#print(contents[['qflag_z', 'qflag_r', 'mis_r_err', 'mis_z_err', 'Sky_r', 'Sky_z']].describe(include='all'))

combined_mask = mask_flags & mask_err & mask_sky
filtered_contents = contents[combined_mask]

# Print out contents to see what's changed
#print(filtered_contents.shape)

In [14]:
r_data = filtered_contents.loc[:, 'mis_r']
z_data = filtered_contents.loc[:, 'mis_z']
i = filtered_contents.loc[:, 'mis_i']
rmz = r_data - z_data

In [15]:
gmi = rmz_gmi_spline(r_data - z_data)
imz = rmz_imz_spline(r_data - z_data)
i_data = imz + z_data
g_data = gmi + i

print(gmi)
print(imz)
print(i_data)
print(g_data)

[1.46252902 1.25975328 1.51309624 ... 0.93853931 1.07895341 1.19694686]
[0.22049803 0.18707828 0.22936808 ... 0.13234399 0.15720893 0.176869  ]
6        17.742768
8        19.450988
12       20.165988
13       19.138619
15       18.232910
           ...    
78259    20.358634
78261    17.583763
78262    18.975964
78266    20.074959
78272    18.599439
Name: mis_z, Length: 34630, dtype: float64
6              NaN
8              NaN
12             NaN
13             NaN
15             NaN
           ...    
78259    21.455494
78261    18.613733
78262    19.942979
78266    21.117273
78272    19.781787
Name: mis_i, Length: 34630, dtype: float64
